# UTD-MHAD: Full-Data Deployment Model (No LOSO)

This notebook trains a **single deployable model** using **all subjects** as training data.
Instead of LOSO cross-validation (which produces fold-specific masks/models for evaluation),
this produces one universal mask + model pair that can classify **any new incoming sample**.

**Approach:**
- Use **all 8 subjects** for training
- Reserve **1 random subject** as validation (for early stopping & metaheuristic fitness)
- Run all 17 feature selection methods (baseline + 3 standard + 14 metaheuristics)
- Train final models and save deployable mask + model pairs
- Benchmark inference timing with actual metaheuristic feature selection from scratch

**Key difference from the LOSO notebook:** No test set exists here — the goal is to
produce the best possible model on all available data for real-world deployment.

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

In [2]:
import torch
print(torch.cuda.device_count())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

1
NVIDIA GeForce RTX 4090


In [3]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import time
import warnings
from pathlib import Path
from tqdm import tqdm
from copy import deepcopy
import random
import sys
from scipy import stats
import psutil
import tracemalloc
import gc
import json as json_lib

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score,
                             precision_score, recall_score, classification_report)

# Standard feature selection imports
from sklearn.feature_selection import mutual_info_classif, RFE, SelectKBest
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

# Import ALL EvoloPy optimizers
sys.path.append('../EvoloPy-master')
from EvoloPy.optimizers import BAT, CS, DE, FFA, GA, GWO, HHO, JAYA, MFO, MVO, PSO, SCA, SSA, WOA

warnings.filterwarnings('ignore')
np.random.seed(42)
torch.manual_seed(42)
random.seed(42)

# Global device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"PyTorch version: {torch.__version__}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Device: cuda
PyTorch version: 2.9.1+cu128
GPU: NVIDIA GeForce RTX 4090


## Configuration

In [4]:
# Paths
FEATURE_DIR = Path("features_new")

# Master output directory — separate from the LOSO results
RESULTS_ROOT = Path("results_all_skel_depth")
RESULTS_ROOT.mkdir(exist_ok=True)
PLOTS_DIR = RESULTS_ROOT / "plots"
PLOTS_DIR.mkdir(exist_ok=True)

# Hyperparameters (same as LOSO notebook)
N_EPOCHS = 300
BATCH_SIZE = 32
LEARNING_RATE = 0.001
HIDDEN_DIM = 128

# Feature selection hyperparameters — metaheuristic
N_POPULATION = 20
MAX_ITERATIONS = 30

# For standard methods (target: select ~50% of features)
TARGET_FEATURE_PERCENTAGE = 0.5

# All EvoloPy optimizer names and their callable entry points
EVOLOPY_OPTIMIZERS = {
    'BAT': BAT.BAT,
    'CS':  CS.CS,
    'DE':  DE.DE,
    'FFA': FFA.FFA,
    'GA':  GA.GA,
    'GWO': GWO.GWO,
    'HHO': HHO.HHO,
    'JAYA': JAYA.JAYA,
    'MFO': MFO.MFO,
    'MVO': MVO.MVO,
    'PSO': PSO.PSO,
    'SCA': SCA.SCA,
    'SSA': SSA.SSA,
    'WOA': WOA.WOA,
}

ALL_METHODS = ['baseline', 'mutual_info', 'rfe', 'lasso'] + [f'meta_{k}' for k in EVOLOPY_OPTIMIZERS.keys()]

# Create per-method subdirectories
for method in ALL_METHODS:
    (RESULTS_ROOT / method).mkdir(exist_ok=True)

print(f"Configuration:")
print(f"  N_EPOCHS: {N_EPOCHS}")
print(f"  Target feature selection: {TARGET_FEATURE_PERCENTAGE*100:.0f}%")
print(f"  Metaheuristic pop: {N_POPULATION}, iter: {MAX_ITERATIONS}")
print(f"  EvoloPy optimizers: {list(EVOLOPY_OPTIMIZERS.keys())}")
print(f"  Total methods (incl. baseline): {len(ALL_METHODS)}")

Configuration:
  N_EPOCHS: 300
  Target feature selection: 50%
  Metaheuristic pop: 20, iter: 30
  EvoloPy optimizers: ['BAT', 'CS', 'DE', 'FFA', 'GA', 'GWO', 'HHO', 'JAYA', 'MFO', 'MVO', 'PSO', 'SCA', 'SSA', 'WOA']
  Total methods (incl. baseline): 18


## Load Data

In [5]:
# Load features
X_feat = joblib.load(FEATURE_DIR / "X_feat.pkl")
y = np.load(FEATURE_DIR / "y.npy")
subjects = np.load(FEATURE_DIR / "subjects.npy")
le = joblib.load(FEATURE_DIR / "label_encoder.pkl")
print(f"Loaded {len(X_feat)} samples")
print(f"Number of classes: {len(np.unique(y))}")
print(f"Number of subjects: {len(np.unique(subjects))}")
# Get feature dimensions
first_sample = X_feat[0]
FEATURE_DIMS = {
'depth': first_sample['depth_feat'].shape[0],
'skeleton': first_sample['skeleton_feat'].shape[0]
}
TOTAL_FEATURES = sum(FEATURE_DIMS.values())
print(f"\nFeature dimensions:")
for modality, dim in FEATURE_DIMS.items():
    print(f"  {modality}: {dim}")
print(f"  TOTAL: {TOTAL_FEATURES}")

Loaded 861 samples
Number of classes: 27
Number of subjects: 8

Feature dimensions:
  depth: 216
  skeleton: 1645
  TOTAL: 1861


## Neural Network Model (Same as Original)

In [6]:
class MultiModalDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.FloatTensor(features)
        self.labels = torch.LongTensor(labels)
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]


class SimpleNN(nn.Module):
    """MLP for unified feature vector (adaptive to feature subset size)"""
    def __init__(self, input_dim, num_classes):
        super().__init__()
        
        # Adaptive hidden layer sizing based on input dimension
        hidden1 = max(128, min(512, input_dim * 2))
        hidden2 = max(64, min(256, hidden1 // 2))
        
        self.classifier = nn.Sequential(
            nn.Linear(input_dim, hidden1),
            nn.BatchNorm1d(hidden1),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(hidden1, hidden2),
            nn.BatchNorm1d(hidden2),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden2, num_classes)
        )
    
    def forward(self, x):
        return self.classifier(x)

print("Neural network model defined")

Neural network model defined


## Helper Functions

In [7]:
def prepare_unified_features(X_feat_list, feature_mask=None):
    """Concatenate all modality features into unified vector"""
    unified_features = []
    
    for sample in X_feat_list:
        feat_vector = np.concatenate([
            sample['depth_feat'],
            sample['skeleton_feat']
        ])
        
        if feature_mask is not None:
            feat_vector = feat_vector[feature_mask]
        
        unified_features.append(feat_vector)
    
    return np.array(unified_features)

def calculate_modality_retention(binary_mask, feature_dims):
    """Calculate how many features retained per modality"""
    start_idx = 0
    retention = {}
    
    for modality, dim in feature_dims.items():
        end_idx = start_idx + dim
        modality_mask = binary_mask[start_idx:end_idx]
        num_selected = np.sum(modality_mask)
        percentage = (num_selected / dim) * 100
        
        retention[modality] = {
            'selected': int(num_selected),
            'total': dim,
            'percentage': percentage
        }
        start_idx = end_idx
    
    return retention

def count_model_parameters(model):
    """Count trainable parameters in a model"""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def get_model_size_mb(model):
    """Get model size in MB"""
    param_size = sum(p.nelement() * p.element_size() for p in model.parameters())
    buffer_size = sum(b.nelement() * b.element_size() for b in model.buffers())
    return (param_size + buffer_size) / (1024 ** 2)

def get_gpu_memory_mb():
    """Get current GPU memory usage in MB"""
    if torch.cuda.is_available():
        return torch.cuda.memory_allocated() / (1024 ** 2)
    return 0.0

def get_dataset_size_mb(X):
    """Get dataset size in MB"""
    return X.nbytes / (1024 ** 2)

print("Helper functions defined")

Helper functions defined


## Training & Evaluation Functions

In [8]:
def train_and_evaluate_deployment(model, train_loader, val_loader, num_epochs, lr, num_classes):
    """
    Train model with early stopping on validation set.
    Since there is no held-out test set in deployment mode,
    we report train accuracy and val accuracy.
    """
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    gpu_mem_before = get_gpu_memory_mb()
    train_start = time.time()

    train_losses = []
    best_val_acc = -1.0
    best_model_state = None
    
    for epoch in range(num_epochs):
        model.train()
        epoch_loss = 0.0

        for features, labels in train_loader:
            features = features.to(DEVICE)
            labels = labels.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(features)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        train_losses.append(epoch_loss / len(train_loader))

        # Check validation accuracy after each epoch
        model.eval()
        val_correct, val_total = 0, 0
        with torch.no_grad():
            for features, labels in val_loader:
                features = features.to(DEVICE)
                outputs = model(features)
                preds = torch.argmax(outputs, dim=1)
                val_correct += (preds.cpu() == labels).sum().item()
                val_total += labels.size(0)

        epoch_val_acc = val_correct / val_total

        if epoch_val_acc > best_val_acc:
            best_val_acc = epoch_val_acc
            best_model_state = deepcopy(model.state_dict())
    
    train_time = time.time() - train_start
    gpu_mem_after = get_gpu_memory_mb()

    # Restore best model
    model.load_state_dict(best_model_state)
    
    # Final evaluation on train and val
    model.eval()
    with torch.no_grad():
        # Train accuracy
        train_preds, train_true = [], []
        for features, labels in train_loader:
            features = features.to(DEVICE)
            outputs = model(features)
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            train_preds.extend(preds)
            train_true.extend(labels.numpy())
        
        # Val accuracy
        val_preds, val_true = [], []
        for features, labels in val_loader:
            features = features.to(DEVICE)
            outputs = model(features)
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            val_preds.extend(preds)
            val_true.extend(labels.numpy())
    
    train_acc = accuracy_score(train_true, train_preds)
    val_acc = accuracy_score(val_true, val_preds)
    
    metrics = {
        'train_acc': train_acc,
        'val_acc': val_acc,
        'val_f1_macro': f1_score(val_true, val_preds, average='macro', zero_division=0),
        'val_f1_weighted': f1_score(val_true, val_preds, average='weighted', zero_division=0),
        'val_precision_macro': precision_score(val_true, val_preds, average='macro', zero_division=0),
        'val_recall_macro': recall_score(val_true, val_preds, average='macro', zero_division=0),
        'val_preds': np.array(val_preds),
        'val_true': np.array(val_true),
        'train_time_sec': train_time,
        'gpu_mem_before_mb': gpu_mem_before,
        'gpu_mem_after_mb': gpu_mem_after,
        'gpu_mem_peak_mb': torch.cuda.max_memory_allocated() / (1024**2) if torch.cuda.is_available() else 0,
        'model_params': count_model_parameters(model),
        'model_size_mb': get_model_size_mb(model),
        'train_losses': train_losses,
    }
    return metrics

print("Deployment training function defined")

Deployment training function defined


## Feature Selection Methods
### 0. EvoloPy Metaheuristics

In [9]:
# ============================================================================
# EXACT EVOLOPY IMPLEMENTATION FROM ORIGINAL CODE
# ============================================================================

def train_model_quick(model, train_loader, val_loader, epochs, lr, device):
    """Quick training for fitness evaluation"""
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    best_val_loss = float('inf')
    best_val_acc = 0.0
    
    for epoch in range(epochs):
        model.train()
        for features, labels in train_loader:
            features, labels = features.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(features), labels)
            loss.backward()
            optimizer.step()
        
        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for features, labels in val_loader:
                features, labels = features.to(device), labels.to(device)
                outputs = model(features)
                val_loss += criterion(outputs, labels).item()
                val_correct += (torch.argmax(outputs, 1) == labels).sum().item()
                val_total += labels.size(0)
        
        val_loss /= len(val_loader)
        val_acc = val_correct / val_total
        if val_loss < best_val_loss:
            best_val_loss, best_val_acc = val_loss, val_acc
    
    return best_val_loss, best_val_acc

# ============================================================================
# FITNESS FUNCTION: Using (1 - accuracy)
# ============================================================================

def create_fitness_function_evolopy(X_train, y_train, X_val, y_val, num_classes):
    """Create fitness function for EvoloPy using (1 - accuracy) + feature penalty"""
    eval_count = [0]
    
    def fitness_function(binary_mask):
        try:
            if binary_mask.dtype != bool:
                binary_mask = binary_mask > 0.5

            num_selected = np.sum(binary_mask)
            if num_selected == 0:
                return 1.0
            
            X_tr_sel = X_train[:, binary_mask]
            X_val_sel = X_val[:, binary_mask]
            
            train_dataset = MultiModalDataset(X_tr_sel, y_train)
            val_dataset = MultiModalDataset(X_val_sel, y_val)
            train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
            val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
            
            model = SimpleNN(X_tr_sel.shape[1], num_classes).to(DEVICE)
            
            val_loss, val_acc = train_model_quick(
                model, train_loader, val_loader,
                epochs=30, lr=1e-3, device=DEVICE
            )
            
            del model, train_dataset, val_dataset, train_loader, val_loader
            torch.cuda.empty_cache()
            
            eval_count[0] += 1
            
            # Weighted fitness: accuracy + feature reduction pressure
            alpha = 0.95
            beta = 0.05
            feature_ratio = num_selected / len(binary_mask)

            fitness = alpha * (1.0 - val_acc) + beta * feature_ratio
            return fitness
            
        except Exception as e:
            print(f"Error in fitness: {e}")
            return 1.0
        
    return fitness_function


def s_transfer(x):
    return 1 / (1 + np.exp(-10 * (x - 0.5)))


def run_evolopy_optimizer(optimizer_name, optimizer_func, X_train, y_train, X_val, y_val, num_classes):
    """Run any EvoloPy optimizer generically"""
    print(f"    Running EvoloPy {optimizer_name}...")
    print(f"      Fitness: (1 - accuracy), Pop: {N_POPULATION}, Iter: {MAX_ITERATIONS}")
    
    fitness_func = create_fitness_function_evolopy(X_train, y_train, X_val, y_val, num_classes)
    
    start = time.time()
    solution = optimizer_func(fitness_func, 0, 1, TOTAL_FEATURES, N_POPULATION, MAX_ITERATIONS)
    exec_time = time.time() - start
    
    binary_mask = np.random.rand(TOTAL_FEATURES) < s_transfer(solution.bestIndividual)
    modality_ret = calculate_modality_retention(binary_mask, FEATURE_DIMS)
    
    results = {
        'mask': binary_mask,
        'convergence': solution.convergence.tolist() if hasattr(solution.convergence, 'tolist') else list(solution.convergence),
        'best_fitness': float(solution.convergence[-1]) if len(solution.convergence) > 0 else float('inf'),
        'execution_time': exec_time,
        'num_selected': int(np.sum(binary_mask)),
        'num_total': len(binary_mask),
        'modality_retention': modality_ret,
        'method': f'meta_{optimizer_name}'
    }
    
    print(f"      Selected: {results['num_selected']}/{results['num_total']} "
          f"({results['num_selected']/results['num_total']*100:.1f}%), "
          f"Fitness: {results['best_fitness']:.4f}, Time: {exec_time:.1f}s")
    print(f"      Modality: D={modality_ret['depth']['percentage']:.0f}%, "
          f"Sk={modality_ret['skeleton']['percentage']:.0f}%, ")
    
    return results

print("All EvoloPy metaheuristic runners defined")
print(f"Available optimizers: {list(EVOLOPY_OPTIMIZERS.keys())}")

All EvoloPy metaheuristic runners defined
Available optimizers: ['BAT', 'CS', 'DE', 'FFA', 'GA', 'GWO', 'HHO', 'JAYA', 'MFO', 'MVO', 'PSO', 'SCA', 'SSA', 'WOA']


### 1-3. Standard Feature Selection Methods (Mutual Info, RFE, LASSO)

In [10]:
def run_mutual_info_fs(X_train, y_train, X_val, y_val, num_classes):
    """Run Mutual Information feature selection"""
    print("    Running Mutual Information...")
    
    start_time = time.time()
    
    mi_scores = mutual_info_classif(X_train, y_train, random_state=42)
    
    k = int(TOTAL_FEATURES * TARGET_FEATURE_PERCENTAGE)
    top_k_indices = np.argsort(mi_scores)[::-1][:k]
    
    binary_mask = np.zeros(TOTAL_FEATURES, dtype=bool)
    binary_mask[top_k_indices] = True
    
    execution_time = time.time() - start_time
    
    modality_retention = calculate_modality_retention(binary_mask, FEATURE_DIMS)
    
    results = {
        'mask': binary_mask,
        'execution_time': execution_time,
        'num_selected': int(np.sum(binary_mask)),
        'num_total': len(binary_mask),
        'modality_retention': modality_retention,
        'mi_scores': mi_scores,
        'method': 'mutual_info'
    }
    
    print(f"      Selected: {results['num_selected']}/{results['num_total']} "
          f"({results['num_selected']/results['num_total']*100:.1f}%), "
          f"Time: {results['execution_time']:.1f}s")
    
    return results


def run_rfe_fs(X_train, y_train, X_val, y_val, num_classes):
    """Run RFE feature selection"""
    print("    Running RFE...")
    
    start_time = time.time()
    
    estimator = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1)
    
    k = int(TOTAL_FEATURES * TARGET_FEATURE_PERCENTAGE)
    selector = RFE(estimator, n_features_to_select=k, step=50)
    selector.fit(X_train, y_train)
    
    binary_mask = selector.support_
    
    execution_time = time.time() - start_time
    
    modality_retention = calculate_modality_retention(binary_mask, FEATURE_DIMS)
    
    results = {
        'mask': binary_mask,
        'execution_time': execution_time,
        'num_selected': int(np.sum(binary_mask)),
        'num_total': len(binary_mask),
        'modality_retention': modality_retention,
        'ranking': selector.ranking_,
        'method': 'rfe'
    }
    
    print(f"      Selected: {results['num_selected']}/{results['num_total']} "
          f"({results['num_selected']/results['num_total']*100:.1f}%), "
          f"Time: {results['execution_time']:.1f}s")
    
    return results


def run_lasso_fs(X_train, y_train, X_val, y_val, num_classes):
    """Run LASSO feature selection"""
    print("    Running LASSO...")
    start_time = time.time()

    lasso = LogisticRegression(
        penalty='l1',
        C=0.01,
        solver='liblinear',
        random_state=42,
        max_iter=1000
    )
    lasso.fit(X_train, y_train)

    coef_abs = np.abs(lasso.coef_).sum(axis=0)

    target_count = int(TOTAL_FEATURES * TARGET_FEATURE_PERCENTAGE)
    top_indices = np.argsort(coef_abs)[::-1][:target_count]
    binary_mask = np.zeros(TOTAL_FEATURES, dtype=bool)
    binary_mask[top_indices] = True

    execution_time = time.time() - start_time
    modality_retention = calculate_modality_retention(binary_mask, FEATURE_DIMS)

    results = {
        'mask': binary_mask,
        'execution_time': execution_time,
        'num_selected': int(np.sum(binary_mask)),
        'num_total': len(binary_mask),
        'modality_retention': modality_retention,
        'coefficients': coef_abs,
        'method': 'lasso'
    }

    print(f"      Selected: {results['num_selected']}/{results['num_total']} "
          f"({results['num_selected']/results['num_total']*100:.1f}%), "
          f"Time: {results['execution_time']:.1f}s")

    return results

print("Standard FS methods defined (Mutual Info, RFE, LASSO)")

Standard FS methods defined (Mutual Info, RFE, LASSO)


## Main Deployment Experiment Runner

In [11]:
def run_all_methods_deployment():
    """
    Run ALL methods on the FULL dataset (no LOSO).
    Uses 1 random subject as validation, all others as training.
    Produces a single deployable mask + model per method.
    """
    print("=" * 80)
    print("STARTING DEPLOYMENT EXPERIMENT (ALL DATA, NO LOSO)")
    print(f"Methods: baseline + 3 standard + {len(EVOLOPY_OPTIMIZERS)} metaheuristics = {len(ALL_METHODS)} total")
    print("=" * 80)
    
    # Prepare unified features
    X_unified = prepare_unified_features(X_feat)
    num_classes = len(np.unique(y))
    
    # =====================================================================
    # TRAIN / VAL SPLIT: Reserve 1 random subject for validation
    # =====================================================================
    all_subjects = np.unique(subjects)
    np.random.seed(42)
    val_subject = np.random.choice(all_subjects)
    
    val_mask = subjects == val_subject
    train_mask = ~val_mask
    
    X_train_raw = X_unified[train_mask]
    y_train = y[train_mask]
    X_val_raw = X_unified[val_mask]
    y_val = y[val_mask]
    
    print(f"\n  Validation subject: {val_subject}")
    print(f"  Train: {len(X_train_raw)} samples ({np.sum(train_mask)} from {len(all_subjects)-1} subjects)")
    print(f"  Val:   {len(X_val_raw)} samples (subject {val_subject})")
    
    # Normalize
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train_raw)
    X_val = scaler.transform(X_val_raw)
    
    # Save scaler for deployment
    joblib.dump(scaler, RESULTS_ROOT / "scaler.pkl")
    print(f"  Scaler saved to {RESULTS_ROOT / 'scaler.pkl'}")
    
    # Original dataset size
    orig_dataset_size_mb = get_dataset_size_mb(X_train) + get_dataset_size_mb(X_val)
    
    # Master results dict
    master_results = {m: {} for m in ALL_METHODS}
    
    # =====================================================================
    # Run each method
    # =====================================================================
    for method_name in ALL_METHODS:
        print(f"\n  --- {method_name.upper()} ---")
        
        fs_start_time = time.time()
        
        # Feature selection
        if method_name == 'baseline':
            feature_mask = np.ones(TOTAL_FEATURES, dtype=bool)
            fs_results = {
                'mask': feature_mask,
                'execution_time': 0,
                'num_selected': TOTAL_FEATURES,
                'num_total': TOTAL_FEATURES,
                'method': 'baseline'
            }
        elif method_name == 'mutual_info':
            fs_results = run_mutual_info_fs(X_train, y_train, X_val, y_val, num_classes)
            feature_mask = fs_results['mask']
        elif method_name == 'rfe':
            fs_results = run_rfe_fs(X_train, y_train, X_val, y_val, num_classes)
            feature_mask = fs_results['mask']
        elif method_name == 'lasso':
            fs_results = run_lasso_fs(X_train, y_train, X_val, y_val, num_classes)
            feature_mask = fs_results['mask']
        elif method_name.startswith('meta_'):
            opt_name = method_name.replace('meta_', '')
            opt_func = EVOLOPY_OPTIMIZERS[opt_name]
            fs_results = run_evolopy_optimizer(opt_name, opt_func, X_train, y_train, X_val, y_val, num_classes)
            feature_mask = fs_results['mask']
        else:
            raise ValueError(f"Unknown method: {method_name}")
        
        fs_time = time.time() - fs_start_time
        
        # Apply feature mask
        X_train_sel = X_train[:, feature_mask]
        X_val_sel = X_val[:, feature_mask]
        
        # Dataset size after selection
        opt_dataset_size_mb = get_dataset_size_mb(X_train_sel) + get_dataset_size_mb(X_val_sel)
        
        # Build and train final model
        model = SimpleNN(X_train_sel.shape[1], num_classes).to(DEVICE)
        
        train_dataset = MultiModalDataset(X_train_sel, y_train)
        val_dataset = MultiModalDataset(X_val_sel, y_val)
        
        train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
        
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()
        
        eval_metrics = train_and_evaluate_deployment(
            model, train_loader, val_loader, N_EPOCHS, LEARNING_RATE, num_classes
        )
        
        # Modality retention
        if method_name != 'baseline':
            modality_ret = fs_results.get('modality_retention', calculate_modality_retention(feature_mask, FEATURE_DIMS))
        else:
            modality_ret = {m: {'selected': d, 'total': d, 'percentage': 100.0} for m, d in FEATURE_DIMS.items()}
        
        # Compile result
        result = {
            'method': method_name,
            'num_train': len(X_train),
            'num_val': len(X_val),
            'val_subject': int(val_subject),
            # Accuracy
            'train_acc': eval_metrics['train_acc'],
            'val_acc': eval_metrics['val_acc'],
            'val_f1_macro': eval_metrics['val_f1_macro'],
            'val_f1_weighted': eval_metrics['val_f1_weighted'],
            'val_precision_macro': eval_metrics['val_precision_macro'],
            'val_recall_macro': eval_metrics['val_recall_macro'],
            # Feature selection info
            'num_features_selected': int(np.sum(feature_mask)),
            'num_features_total': TOTAL_FEATURES,
            'feature_retention_pct': float(np.sum(feature_mask) / TOTAL_FEATURES * 100),
            'modality_retention': modality_ret,
            'feature_mask': feature_mask,
            # Timing
            'fs_execution_time': fs_results.get('execution_time', 0),
            'train_time_sec': eval_metrics['train_time_sec'],
            'total_time_sec': fs_time + eval_metrics['train_time_sec'],
            # Model size
            'model_params': eval_metrics['model_params'],
            'model_size_mb': eval_metrics['model_size_mb'],
            # Dataset size
            'original_dataset_size_mb': orig_dataset_size_mb,
            'optimized_dataset_size_mb': opt_dataset_size_mb,
            'dataset_reduction_pct': float((1 - opt_dataset_size_mb / orig_dataset_size_mb) * 100) if orig_dataset_size_mb > 0 else 0,
            # GPU
            'gpu_mem_peak_mb': eval_metrics['gpu_mem_peak_mb'],
            # Convergence (metaheuristics only)
            'convergence': fs_results.get('convergence', None),
            'best_fitness': fs_results.get('best_fitness', None),
        }
        
        master_results[method_name] = result
        
        # Save mask and model
        method_dir = RESULTS_ROOT / method_name
        np.save(method_dir / "deployment_mask.npy", feature_mask)
        torch.save(model.state_dict(), method_dir / "deployment_model.pth")
        
        # Save result JSON (exclude large arrays)
        result_save = {k: v for k, v in result.items() if k not in ['feature_mask']}
        result_clean = {}
        for k, v in result_save.items():
            if isinstance(v, np.ndarray):
                result_clean[k] = v.tolist()
            elif isinstance(v, (np.floating, np.integer)):
                result_clean[k] = float(v)
            elif isinstance(v, dict):
                result_clean[k] = {}
                for kk, vv in v.items():
                    if isinstance(vv, dict):
                        result_clean[k][kk] = {kkk: float(vvv) if isinstance(vvv, (np.floating, np.integer)) else vvv for kkk, vvv in vv.items()}
                    else:
                        result_clean[k][kk] = float(vv) if isinstance(vv, (np.floating, np.integer)) else vv
            else:
                result_clean[k] = v
        
        with open(method_dir / "deployment_result.json", 'w') as f:
            json_lib.dump(result_clean, f, indent=2, default=str)
        
        print(f"    Train Acc: {eval_metrics['train_acc']*100:.2f}%, "
              f"Val Acc: {eval_metrics['val_acc']*100:.2f}%, "
              f"F1: {eval_metrics['val_f1_macro']*100:.2f}%, "
              f"Features: {np.sum(feature_mask)}/{TOTAL_FEATURES} "
              f"({np.sum(feature_mask)/TOTAL_FEATURES*100:.1f}%), "
              f"Model: {eval_metrics['model_size_mb']:.3f}MB")
        print(f"    Saved: deployment_mask.npy, deployment_model.pth, deployment_result.json")
        
        # Cleanup
        del model, train_dataset, val_dataset
        del train_loader, val_loader
        torch.cuda.empty_cache()
        gc.collect()
    
    print(f"\n{'='*80}")
    print("DEPLOYMENT EXPERIMENT COMPLETED")
    print(f"{'='*80}")
    
    return master_results

print("Deployment experiment runner defined")

Deployment experiment runner defined


## Run All Experiments

In [12]:
master_results = run_all_methods_deployment()

STARTING DEPLOYMENT EXPERIMENT (ALL DATA, NO LOSO)
Methods: baseline + 3 standard + 14 metaheuristics = 18 total

  Validation subject: 7
  Train: 753 samples (753 from 7 subjects)
  Val:   108 samples (subject 7)
  Scaler saved to results_all_skel_depth/scaler.pkl

  --- BASELINE ---
    Train Acc: 100.00%, Val Acc: 94.44%, F1: 92.73%, Features: 1861/1861 (100.0%), Model: 4.176MB
    Saved: deployment_mask.npy, deployment_model.pth, deployment_result.json

  --- MUTUAL_INFO ---
    Running Mutual Information...
      Selected: 930/1861 (50.0%), Time: 10.5s
    Train Acc: 100.00%, Val Acc: 98.15%, F1: 98.12%, Features: 930/1861 (50.0%), Model: 2.358MB
    Saved: deployment_mask.npy, deployment_model.pth, deployment_result.json

  --- RFE ---
    Running RFE...
      Selected: 930/1861 (50.0%), Time: 1.9s
    Train Acc: 100.00%, Val Acc: 97.22%, F1: 97.18%, Features: 930/1861 (50.0%), Model: 2.358MB
    Saved: deployment_mask.npy, deployment_model.pth, deployment_result.json

  --- LASS

## Build Results CSV

In [13]:
rows = []

for method_name in ALL_METHODS:
    if method_name not in master_results or not master_results[method_name]:
        continue
    r = master_results[method_name]
    
    row = {
        'Method': method_name,
        'Train Accuracy (%)': r['train_acc'] * 100,
        'Val Accuracy (%)': r['val_acc'] * 100,
        'F1 Macro (%)': r['val_f1_macro'] * 100,
        'F1 Weighted (%)': r['val_f1_weighted'] * 100,
        'Precision Macro (%)': r['val_precision_macro'] * 100,
        'Recall Macro (%)': r['val_recall_macro'] * 100,
        'Features Selected': r['num_features_selected'],
        'Features Total': r['num_features_total'],
        'Feature Retention (%)': r['feature_retention_pct'],
        'FS Time (s)': r['fs_execution_time'],
        'Train Time (s)': r['train_time_sec'],
        'Total Time (s)': r['total_time_sec'],
        'Model Params': r['model_params'],
        'Model Size (MB)': r['model_size_mb'],
        'Original Dataset (MB)': r['original_dataset_size_mb'],
        'Optimized Dataset (MB)': r['optimized_dataset_size_mb'],
        'Dataset Reduction (%)': r['dataset_reduction_pct'],
        'GPU Peak (MB)': r['gpu_mem_peak_mb'],
    }
    
    # Add per-modality retention
    for mod_name in FEATURE_DIMS.keys():
        mod_ret = r['modality_retention'].get(mod_name, {})
        if isinstance(mod_ret, dict):
            row[f'{mod_name}_retention (%)'] = mod_ret.get('percentage', 0)
            row[f'{mod_name}_selected'] = mod_ret.get('selected', 0)
        else:
            row[f'{mod_name}_retention (%)'] = float(mod_ret)
    
    rows.append(row)

df_all = pd.DataFrame(rows)
df_all = df_all.round(4)

csv_path = RESULTS_ROOT / "deployment_results.csv"
df_all.to_csv(csv_path, index=False)
print(f"Saved: {csv_path}")
print(f"Shape: {df_all.shape}")
print(df_all.to_string())

Saved: results_all_skel_depth/deployment_results.csv
Shape: (18, 23)
         Method  Train Accuracy (%)  Val Accuracy (%)  F1 Macro (%)  F1 Weighted (%)  Precision Macro (%)  Recall Macro (%)  Features Selected  Features Total  Feature Retention (%)  FS Time (s)  Train Time (s)  Total Time (s)  Model Params  Model Size (MB)  Original Dataset (MB)  Optimized Dataset (MB)  Dataset Reduction (%)  GPU Peak (MB)  depth_retention (%)  depth_selected  skeleton_retention (%)  skeleton_selected
0      baseline            100.0000           94.4444       92.7337          92.7337              91.6049           94.4444               1861            1861               100.0000       0.0000          8.7330          8.7330       1093147           4.1759                12.2247                 12.2247                 0.0000        42.6338             100.0000             216                100.0000               1645
1   mutual_info            100.0000           98.1481       98.1188          98.1188 

## Visualizations

In [14]:
# PLOT 1: Validation Accuracy for all methods
fig, ax = plt.subplots(figsize=(16, 7))
methods_sorted = df_all.sort_values('Val Accuracy (%)', ascending=False)['Method'].tolist()

bars = ax.bar(range(len(methods_sorted)), 
              df_all.set_index('Method').loc[methods_sorted, 'Val Accuracy (%)'],
              alpha=0.7, color='steelblue')
ax.set_xticks(range(len(methods_sorted)))
ax.set_xticklabels(methods_sorted, rotation=60, ha='right')
ax.set_ylabel('Validation Accuracy (%)', fontsize=12)
ax.set_title('Deployment Model: Validation Accuracy by Method', fontsize=16)
ax.grid(axis='y', alpha=0.3)

for i, bar in enumerate(bars):
    val = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., val + 0.3,
            f'{val:.1f}', ha='center', va='bottom', fontsize=8, fontweight='bold')

plt.tight_layout()
plt.savefig(PLOTS_DIR / '01_deployment_val_accuracy.png', dpi=300, bbox_inches='tight')
plt.close()
print("Saved: 01_deployment_val_accuracy.png")

Saved: 01_deployment_val_accuracy.png


In [15]:
# PLOT 2: Feature Retention per method
fig, ax = plt.subplots(figsize=(16, 7))
feat_data = df_all.set_index('Method').loc[ALL_METHODS, 'Feature Retention (%)']
bars = ax.bar(range(len(ALL_METHODS)), feat_data.values, alpha=0.7, color='coral')
ax.set_xticks(range(len(ALL_METHODS)))
ax.set_xticklabels(ALL_METHODS, rotation=60, ha='right')
ax.axhline(y=100, color='r', linestyle='--', label='All Features (100%)')
ax.set_ylabel('Feature Retention (%)', fontsize=12)
ax.set_title('Deployment: Feature Retention by Method', fontsize=16)
ax.legend()
ax.grid(axis='y', alpha=0.3)

for bar, val in zip(bars, feat_data.values):
    if not np.isnan(val):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1,
                f'{val:.1f}%', ha='center', va='bottom', fontsize=8, fontweight='bold')

plt.tight_layout()
plt.savefig(PLOTS_DIR / '02_deployment_feature_retention.png', dpi=300, bbox_inches='tight')
plt.close()
print("Saved: 02_deployment_feature_retention.png")

Saved: 02_deployment_feature_retention.png


In [16]:
# PLOT 3: Accuracy vs Feature Retention trade-off
fig, ax = plt.subplots(figsize=(12, 8))

for _, row in df_all.iterrows():
    method = row['Method']
    ax.scatter(row['Feature Retention (%)'], row['Val Accuracy (%)'], s=100, alpha=0.8)
    ax.annotate(method, (row['Feature Retention (%)'], row['Val Accuracy (%)']),
                textcoords="offset points", xytext=(5, 5), fontsize=7)

ax.set_xlabel('Feature Retention (%)', fontsize=12)
ax.set_ylabel('Validation Accuracy (%)', fontsize=12)
ax.set_title('Deployment: Accuracy vs Feature Retention Trade-off', fontsize=16)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(PLOTS_DIR / '03_deployment_acc_vs_retention.png', dpi=300, bbox_inches='tight')
plt.close()
print("Saved: 03_deployment_acc_vs_retention.png")

Saved: 03_deployment_acc_vs_retention.png


In [17]:
# PLOT 4: Per-modality retention heatmap
modality_data = {}
for method in ALL_METHODS:
    modality_data[method] = {}
    for mod in FEATURE_DIMS.keys():
        col = f'{mod}_retention (%)'
        if col in df_all.columns:
            mdf = df_all[df_all['Method'] == method]
            modality_data[method][mod] = mdf[col].values[0] if len(mdf) > 0 else 0

mod_df = pd.DataFrame(modality_data).T
mod_df = mod_df.reindex(ALL_METHODS)

fig, ax = plt.subplots(figsize=(10, 10))
sns.heatmap(mod_df, annot=True, fmt='.1f', cmap='RdYlGn', ax=ax, linewidths=0.5,
            vmin=0, vmax=100, cbar_kws={'label': 'Retention (%)'})
ax.set_title('Deployment: Modality-wise Feature Retention (%)', fontsize=14)
ax.set_ylabel('Method')
ax.set_xlabel('Modality')
plt.tight_layout()
plt.savefig(PLOTS_DIR / '04_deployment_modality_retention.png', dpi=300, bbox_inches='tight')
plt.close()
print("Saved: 04_deployment_modality_retention.png")

Saved: 04_deployment_modality_retention.png


In [18]:
# PLOT 5: Execution time comparison
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

fs_data = df_all.set_index('Method').loc[ALL_METHODS, 'FS Time (s)']
axes[0].bar(range(len(ALL_METHODS)), fs_data.values, alpha=0.7, color='steelblue')
axes[0].set_xticks(range(len(ALL_METHODS)))
axes[0].set_xticklabels(ALL_METHODS, rotation=60, ha='right')
axes[0].set_ylabel('Feature Selection Time (s)', fontsize=12)
axes[0].set_title('Feature Selection Time', fontsize=14)
axes[0].grid(axis='y', alpha=0.3)

total_data = df_all.set_index('Method').loc[ALL_METHODS, 'Total Time (s)']
axes[1].bar(range(len(ALL_METHODS)), total_data.values, alpha=0.7, color='darkorange')
axes[1].set_xticks(range(len(ALL_METHODS)))
axes[1].set_xticklabels(ALL_METHODS, rotation=60, ha='right')
axes[1].set_ylabel('Total Time (s)', fontsize=12)
axes[1].set_title('Total Time (FS + Training)', fontsize=14)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(PLOTS_DIR / '05_deployment_execution_time.png', dpi=300, bbox_inches='tight')
plt.close()
print("Saved: 05_deployment_execution_time.png")

Saved: 05_deployment_execution_time.png


In [19]:
# PLOT 6: Metaheuristic convergence curves
meta_methods = [m for m in ALL_METHODS if m.startswith('meta_')]

if len(meta_methods) > 0:
    fig, ax = plt.subplots(figsize=(14, 8))
    
    for method in meta_methods:
        if method in master_results and master_results[method]:
            conv = master_results[method].get('convergence', None)
            if conv is not None:
                ax.plot(conv, label=method.replace('meta_', ''), linewidth=1.5)
    
    ax.set_xlabel('Iteration', fontsize=12)
    ax.set_ylabel('Fitness (1 - Accuracy)', fontsize=12)
    ax.set_title('Deployment: Metaheuristic Convergence Curves', fontsize=14)
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / '06_deployment_convergence.png', dpi=300, bbox_inches='tight')
    plt.close()
    print("Saved: 06_deployment_convergence.png")

Saved: 06_deployment_convergence.png


## Final Summary

In [20]:
print("\n" + "=" * 80)
print("DEPLOYMENT EXPERIMENT COMPLETE - RESULTS SUMMARY")
print("=" * 80)

print(f"\nOutput directory: {RESULTS_ROOT}")
print(f"\nPer-method folders (each contains deployment_mask.npy + deployment_model.pth):")
for method in ALL_METHODS:
    method_dir = RESULTS_ROOT / method
    n_files = len(list(method_dir.glob('*')))
    print(f"  {method_dir}/ ({n_files} files)")

print(f"\nPlots directory: {PLOTS_DIR}")
for p in sorted(PLOTS_DIR.glob('*.png')):
    print(f"  {p.name}")

# Print top 5 methods by val accuracy
print("\nTOP 5 METHODS BY VALIDATION ACCURACY:")
top5 = df_all.nlargest(5, 'Val Accuracy (%)')
for _, row in top5.iterrows():
    print(f"  {row['Method']:20s}: {row['Val Accuracy (%)']:.2f}% "
          f"(Features: {row['Feature Retention (%)']:.1f}%)")

# Identify best metaheuristic
meta_df_all = df_all[df_all['Method'].str.startswith('meta_')]
if len(meta_df_all) > 0:
    best_meta = meta_df_all.loc[meta_df_all['Val Accuracy (%)'].idxmax()]
    print(f"\nBEST METAHEURISTIC: {best_meta['Method']}")
    print(f"  Val Accuracy: {best_meta['Val Accuracy (%)']:.2f}%")
    print(f"  Features: {int(best_meta['Features Selected'])}/{int(best_meta['Features Total'])} "
          f"({best_meta['Feature Retention (%)']:.1f}%)")

print(f"\n{'='*80}")
print("DEPLOYABLE ARTIFACTS (per method):")
print(f"  - deployment_mask.npy   : Feature selection mask")
print(f"  - deployment_model.pth  : Trained PyTorch model weights")
print(f"  - deployment_result.json: Full metrics and config")
print(f"  - scaler.pkl            : StandardScaler (shared, in root)")
print(f"{'='*80}")


DEPLOYMENT EXPERIMENT COMPLETE - RESULTS SUMMARY

Output directory: results_all_skel_depth

Per-method folders (each contains deployment_mask.npy + deployment_model.pth):
  results_all_skel_depth/baseline/ (3 files)
  results_all_skel_depth/mutual_info/ (3 files)
  results_all_skel_depth/rfe/ (3 files)
  results_all_skel_depth/lasso/ (3 files)
  results_all_skel_depth/meta_BAT/ (3 files)
  results_all_skel_depth/meta_CS/ (3 files)
  results_all_skel_depth/meta_DE/ (3 files)
  results_all_skel_depth/meta_FFA/ (3 files)
  results_all_skel_depth/meta_GA/ (3 files)
  results_all_skel_depth/meta_GWO/ (3 files)
  results_all_skel_depth/meta_HHO/ (3 files)
  results_all_skel_depth/meta_JAYA/ (3 files)
  results_all_skel_depth/meta_MFO/ (3 files)
  results_all_skel_depth/meta_MVO/ (3 files)
  results_all_skel_depth/meta_PSO/ (3 files)
  results_all_skel_depth/meta_SCA/ (3 files)
  results_all_skel_depth/meta_SSA/ (3 files)
  results_all_skel_depth/meta_WOA/ (3 files)

Plots directory: results